In [23]:
import pandas as pd
import json
import statsmodels.api as sm
from statsmodels.formula.api import ols

In [30]:
with open(f"../responses/result_eval.json", 'r') as result_file:

    results = json.load(result_file)
results_df = pd.DataFrame(results)
multi_index_results_df = results_df.set_index(['embedder', 'generator', 'distance_metric']).sort_index()
multi_index_results_df

context_utilization  \
embedder                       generator                     distance_metric                        
BM25                           Ministral-3-14B-Instruct-2512 nan                         0.333333   
                                                             nan                         0.500000   
                                                             nan                         1.000000   
                                                             nan                         0.583333   
                                                             nan                              NaN   
...                                                                                           ...   
multilingual-e5-large-instruct Qwen3-30B-A3B-Instruct-2507   manhattan                   0.000000   
                                                             manhattan                   0.000000   
                                                             manhattan                   0.000000   
                                                             manhattan                   0.000000   
                                                             manhattan                   0.000000   

                                                                              answer_relevancy  \
embedder                       generator                     distance_metric                     
BM25                           Ministral-3-14B-Instruct-2512 nan                      0.000000   
                                                             nan                      0.000000   
                                                             nan                      0.563342   
                                                             nan                      0.000000   
                                                             nan                      0.000000   
...                                                                                        ...   
multilingual-e5-large-instruct Qwen3-30B-A3B-Instruct-2507   manhattan                     NaN   
                                                             manhattan                     NaN   
                                                             manhattan                     NaN   
                                                             manhattan                     NaN   
                                                             manhattan                     NaN   

                                                                              faithfullness  \
embedder                       generator                     distance_metric                  
BM25                           Ministral-3-14B-Instruct-2512 nan                        NaN   
                                                             nan                        NaN   
                                                             nan                        NaN   
                                                             nan                        NaN   
                                                             nan                        NaN   
...                                                                                     ...   
multilingual-e5-large-instruct Qwen3-30B-A3B-Instruct-2507   manhattan                  NaN   
                                                             manhattan                  NaN   
                                                             manhattan                  NaN   
                                                             manhattan                  NaN   
                                                             manhattan                  NaN   

                                                                                              user_prompt  \
embedder                       generator                     distance_metric                                
BM25                           Mini

In [21]:
metrics = ['mean', 'std']

grouped = results_df.groupby(['embedder', 'generator', 'distance_metric'], dropna=False).agg({'context_utilization' : metrics,
                                                                                              'answer_relevancy' : metrics,
                                                                                              'faithfullness' : metrics})
grouped

context_utilization  \
                                                                                                       mean   
embedder                       generator                              distance_metric                         
BM25                           Ministral-3-14B-Instruct-2512          NaN                          0.557971   
                               NVIDIA-Nemotron-3-Super-120B-A12B-BF16 NaN                          0.345833   
                               Qwen3-30B-A3B-Instruct-2507            NaN                          0.013333   
Qwen3-Embedding-8B             Ministral-3-14B-Instruct-2512          cosine_similarity            0.040000   
                                                                      manhattan                    0.100000   
                               NVIDIA-Nemotron-3-Super-120B-A12B-BF16 cosine_similarity            0.068182   
                                                                      manhattan                    0.142857   
                               Qwen3-30B-A3B-Instruct-2507            cosine_similarity            0.000000   
                                                                      manhattan                    0.000000   
bge-m3                         Ministral-3-14B-Instruct-2512          cosine_similarity            0.000000   
                                                                      manhattan                    0.203333   
                               NVIDIA-Nemotron-3-Super-120B-A12B-BF16 cosine_similarity            0.087963   
                                                                      manhattan                    0.436508   
                               Qwen3-30B-A3B-Instruct-2507            cosine_similarity            0.000000   
                                                                      manhattan                    0.000000   
multilingual-e5-large-instruct Ministral-3-14B-Instruct-2512          cosine_similarity            0.000000   
                                                                      manhattan                    0.196667   
                               NVIDIA-Nemotron-3-Super-120B-A12B-BF16 cosine_similarity            0.057018   
                                                                      manhattan                    0.228261   
                               Qwen3-30B-A3B-Instruct-2507            cosine_similarity            0.000000   
                                                                      manhattan                    0.000000   

                                                                                                   \
                                                                                              std   
embedder                       generator                              distance_metric               
BM25                           Ministral-3-14B-Instruct-2512          NaN                0.353048   
                               NVIDIA-Nemotron-3-Super-120B-A12B-BF16 NaN                0.357946   
                               Qwen3-30B-A3B-Instruct-2507            NaN                0.066667   
Qwen3-Embedding-8B             Ministral-3-14B-Instruct-2512          cosine_similarity  0.200000   
                                                                      manhattan          0.235702   
                               NVIDIA-Nemotron-3-Super-120B-A12B-BF16 cosine_similarity  0.233781   
                                                                      manhattan          0.358569   
                               Qwen3-30B-A3B-Instruct-2507            cosine_similarity  0.000000   
                                                                      manhattan          0.000000   
bge-m3                         Ministral-3-14B-Instruct-2512          cosine_similarity  0.000000   
                                                                      manhattan          0.315568   
         

In [33]:
# Handle missing values in categorical variables (e.g., BM25 doesn't have a distance metric)
results_df['distance_metric'] = results_df['distance_metric'].fillna('None')

metrics = ['answer_relevancy', 'context_utilization']

for metric in metrics:
    # Handle missing values in your target variable if necessary
    anova_df = results_df.dropna(subset=[metric])

    # Define and fit the Ordinary Least Squares (OLS) model
    # Wrap categorical variables in C() to tell statsmodels they are factors
    formula = f'{metric} ~ C(embedder) + C(generator) + C(distance_metric)'
    model = ols(formula, data=anova_df).fit()

    # Perform a Type II ANOVA (recommended for unbalanced/unequal group sizes)
    anova_table = sm.stats.anova_lm(model, typ=2)

    print(f'ANOVA of {metric}')
    
    # Display the results
    print(anova_table)
    print()

ANOVA of answer_relevancy
                       sum_sq     df          F    PR(>F)
C(embedder)          0.102648    3.0   0.582293  0.626980
C(generator)         1.176225    1.0  20.017220  0.000010
C(distance_metric)   0.081426    2.0   0.692866  0.500839
Residual            20.154897  343.0        NaN       NaN

ANOVA of context_utilization
                       sum_sq     df          F        PR(>F)
C(embedder)          0.033662    3.0   0.198966  8.196454e-01
C(generator)         3.463110    2.0  30.704160  2.811265e-13
C(distance_metric)   0.044882    2.0   0.397931  5.284586e-01
Residual            27.182295  482.0        NaN           NaN



/Users/maritvandenhelder/miniconda3/envs/thesis/lib/python3.14/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 3, but rank is 2
  warnings.warn('covariance of constraints does not have full '
/Users/maritvandenhelder/miniconda3/envs/thesis/lib/python3.14/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 2, but rank is 1
  warnings.warn('covariance of constraints does not have full '
